# Project 1: emotion classification on ISEAR

Comparing 3 representations × 3 architectures with cross-validation and a Bayesian ROPE analysis, reusing the tutorial code in `part1/`.

| Section | Who |
| --- | --- |
| 1. Data, 2. Folds, 3. Tokeniser | Britt |
| 4. Representations | Zeno |
| 5. Architectures | Armani |
| 6. Experiments, 7. Bayesian comparison, 8. Error analysis | Together |

Run `python explore_data.py` once first: it creates `data/isear_clean.csv`.

In [20]:
from pathlib import Path

import numpy as np
import pandas as pd

import part1
from part1.clslib import NamedClasses

results_folder = Path("results/project1")
results_folder.mkdir(parents=True, exist_ok=True)

## 1. Data
Cleaned ISEAR (see `explore_data.py`): 7,516 raw texts, 84 removed (10 placeholders, 21 same text with conflicting labels, 53 duplicates), 7,432 left. The classes are balanced, so accuracy is a fair metric (majority baseline 14.5%).

In [21]:
LABELS = ["joy", "fear", "anger", "sadness", "disgust", "shame", "guilt"]
outcome_space = NamedClasses(LABELS)

df = pd.read_csv("data/isear_clean.csv")
pairs = list(zip(df["text"], df["label_id"]))  # (text, label) pairs, as in the tutorial
print(len(pairs), "texts")
df["label"].value_counts()

7432 texts


label
joy        1079
anger      1077
fear       1070
disgust    1060
shame      1053
sadness    1050
guilt      1043
Name: count, dtype: int64

## 2. Cross-validation folds
5 folds, repeated with 3 seeds (15 accuracies per system), using the tutorial's `cross_validation_split`. Every system uses these same folds, so the scores are paired for the Bayesian comparison.

In [22]:
seeds = [0, 3, 5]
num_folds = 5
cv_sets = [part1.experiment.cross_validation_split(pairs, k=num_folds, seed=seed) for seed in seeds]

# check: are the classes spread evenly over the folds?
for fold, split in cv_sets[0].items():
    counts = pd.Series([label for _, label in split["validation"]]).value_counts().sort_index()
    print(f"fold {fold}: {len(split['validation'])} texts, per class {counts.tolist()}")

fold 0: 1487 texts, per class [231, 206, 186, 239, 219, 210, 196]
fold 1: 1487 texts, per class [198, 217, 227, 199, 215, 208, 223]
fold 2: 1486 texts, per class [237, 186, 228, 202, 221, 219, 193]
fold 3: 1486 texts, per class [207, 221, 219, 207, 206, 216, 210]
fold 4: 1486 texts, per class [206, 240, 217, 203, 199, 200, 221]


## 3. Tokeniser
BPE tokeniser trained with the tutorial's `train_tokenizer` on the ISEAR texts themselves (~165k words is enough for a small vocabulary). It uses no labels, so training on all texts does not leak label information (the tutorial does the same).

To choose the vocabulary size, we compare how finely each size splits words: a good size keeps frequent words, such as emotion words, whole without making the vocabulary huge.

In [23]:
texts = [text for text, _ in pairs]
n_words = sum(len(t.split()) for t in texts)
example = "I felt ashamed when I shouted at my girlfriend."

for size in [500, 1000, 2000, 5000]:
    tok = part1.experiment.train_tokenizer(texts, size, min_frequency=2)
    n_tokens = sum(len(tok.encode(t).ids) for t in texts)
    print(f"{size:>5}: {n_tokens / n_words:.2f} tokens per word | {tok.encode(example).tokens}")




  500: 2.10 tokens per word | ['▁I', '▁felt', '▁as', 'ham', 'ed', '▁when', '▁I', '▁sh', 'ou', 'ted', '▁at', '▁my', '▁girl', 'friend', '.']



 1000: 1.74 tokens per word | ['▁I', '▁felt', '▁ashamed', '▁when', '▁I', '▁sh', 'ou', 'ted', '▁at', '▁my', '▁girlfriend', '.']



 2000: 1.50 tokens per word | ['▁I', '▁felt', '▁ashamed', '▁when', '▁I', '▁sh', 'ou', 'ted', '▁at', '▁my', '▁girlfriend.']



 5000: 1.28 tokens per word | ['▁I', '▁felt', '▁ashamed', '▁when', '▁I', '▁shouted', '▁at', '▁my', '▁girlfriend.']


In [24]:
vocab_size = 1000  # same as tutorial (500 is too small and with 2000 puctuation gets glued on the words)
tokenizer = part1.experiment.train_tokenizer(texts, vocab_size, min_frequency=2)
tokenizer.save(f"{results_folder}/bpe_vs{vocab_size}.json")
print(tokenizer.encode(example).tokens)




['▁I', '▁felt', '▁ashamed', '▁when', '▁I', '▁sh', 'ou', 'ted', '▁at', '▁my', '▁girlfriend', '.']


## 4. Representations (Zeno)
- token indicators (tutorial)
- bag-of-words (tutorial exercise)
- one extra of our choice

In [32]:
from collections import Counter
from tokenizers import Tokenizer

class Encoder:

    def __len__(self) -> int:
        """Dimensionality of the representation space"""
        raise NotImplementedError("Concrete implementations will be discussed later")

    def __call__(self, x: str) -> 'np.array':
        """Represent x as a D-dimensional vector of real numbers"""
        raise NotImplementedError("Concrete implementations will be discussed later")


class TokenIndicator(Encoder):
    """h_v(x) = 1 if BPE token v occurs in x, else 0 (tutorial)."""

    def __init__(self, tokenizer: Tokenizer):
        self.tokenizer = tokenizer
        
    def __len__(self) -> int:
        """Dimensionality of the representation space"""
        return self.tokenizer.get_vocab_size()

    def __call__(self, x: str) -> 'np.array':
        """Represent x as a D-dimensional vector of real numbers"""
        unique_token_ids = list(set(self.tokenizer.encode(x).ids))

        # Set the 0 vector to 1 in every word that appears
        vector = np.zeros(len(self))
        vector[unique_token_ids] = 1
        return vector


class BagOfWords(Encoder):
    """h_v(x) = number of times BPE token v occurs in x (tutorial exercise)."""

    def __init__(self, tokenizer: Tokenizer):
        self.tokenizer = tokenizer
        self.dim = tokenizer.get_vocab_size()

    def __len__(self) -> int:
        return self.dim

    def __call__(self, x: str) -> np.ndarray:
        vector = np.zeros(self.dim)
        # np.add.at accumulates repeated ids; vector[ids] += 1 would count each id only once
        np.add.at(vector, self.tokenizer.encode(x).ids, 1)
        return vector


class BagOfUnigramsAndBigrams(Encoder):
    """
    Counts of BPE tokens plus counts of adjacent token pairs (the tutorial's 'bag of bigrams').
    Only bigrams seen at least `min_count` times in `texts` get a dimension, otherwise D ~ V^2.
    """

    def __init__(self, tokenizer: Tokenizer, texts: list, min_count: int = 2):
        self.tokenizer = tokenizer
        self.num_unigrams = tokenizer.get_vocab_size()
        bigram_counts = Counter(
            pair
            for text in texts
            for pair in self._bigrams(tokenizer.encode(text).ids)
        )
        kept = sorted(pair for pair, count in bigram_counts.items() if count >= min_count)
        # bigram dimensions come after the V unigram dimensions
        self.bigram_index = {pair: self.num_unigrams + i for i, pair in enumerate(kept)}
        self.dim = self.num_unigrams + len(self.bigram_index)

    @staticmethod
    def _bigrams(ids):
        return zip(ids, ids[1:])

    def __len__(self) -> int:
        return self.dim

    def __call__(self, x: str) -> np.ndarray:
            ids = self.tokenizer.encode(x).ids
            vector = np.zeros(self.dim)
            np.add.at(vector, ids, 1)
            bigram_dims = [self.bigram_index[p] for p in self._bigrams(ids) if p in self.bigram_index]
            np.add.at(vector, bigram_dims, 1)  # unseen bigrams are simply dropped
            return vector
    
    """FOR SECTION 8!
    def coords(self):
        Readable axis labels, e.g. for inspecting learned weights in the error analysis.

        id_to_token = {i: t for t, i in self.tokenizer.get_vocab().items()}
        labels = [id_to_token[i] for i in range(self.num_unigrams)]
        labels += [id_to_token[a] + " " + id_to_token[b] for (a, b) in self.bigram_index]
        return labels
    """
    

## 5. Architectures (Armani)
Log-linear and 1-hidden-layer MLP, plus one extra architecture

In [33]:
import scipy.sparse as sp

def softmax_rows(S):
    E = np.exp(S - S.max(axis=1, keepdims=True))
    return E / E.sum(axis=1, keepdims=True)

def one_hot(y, K):
    Y = np.zeros((len(y), K))
    Y[np.arange(len(y)), y] = 1
    return Y

def encode_all(encoder, texts):
    # sparse: the bigram matrix would be several GB dense
    return sp.vstack([sp.csr_matrix(encoder(t)) for t in texts], format="csr")


class LogLinear:
    def __init__(self, encoder, K=7):
        self.encoder, self.K = encoder, K

    def train(self, X, y, lr=0.1, reg_strength=1e-3, max_epochs=20000, tol=1e-5):
        n, Y = X.shape[0], one_hot(y, self.K)
        self.W = np.zeros((self.K, X.shape[1]))
        prev = np.inf
        for epoch in range(max_epochs):
            P = softmax_rows(X @ self.W.T)
            # objective these updates minimize: mean NLL + L2 penalty
            loss = (-np.mean(np.log(P[np.arange(n), y] + 1e-12))
                    + reg_strength / 2 * np.sum(self.W ** 2))
            if prev - loss < tol:          # converged: loss stopped improving
                break
            prev = loss
            grad = (X.T @ (Y - P)).T       # = sum over samples of (y - p) x^T
            self.W += lr * (grad / n - reg_strength * self.W)
        self.epochs_run = epoch + 1

    def predict_ids(self, X):
        return np.argmax(X @ self.W.T, axis=1)

    def predict(self, text):
        return LABELS[int(self.predict_ids(sp.csr_matrix(self.encoder(text)))[0])]


class MLP:
    """hidden=(100,) is the tutorial MLP; longer tuples give deeper nets.
    init_scale=None uses He init, a number uses fixed-std init."""

    def __init__(self, encoder, hidden=(100,), K=7, init_scale=None, seed=42):
        self.encoder, self.hidden, self.K = encoder, list(hidden), K
        self.init_scale = init_scale
        self.rng = np.random.default_rng(seed)

    def forward(self, X):
        hs, zs = [X], []
        for i, (W, b) in enumerate(zip(self.Ws, self.bs)):
            z = hs[-1] @ W.T + b
            zs.append(z)
            if i < len(self.Ws) - 1:
                hs.append(np.maximum(z, 0))
        return zs, hs, softmax_rows(zs[-1])

    def train(self, X, y, lr=0.1, reg_strength=1e-3, max_epochs=5000, tol=1e-5):
        n, Y = X.shape[0], one_hot(y, self.K)
        sizes = [X.shape[1]] + self.hidden + [self.K]
        self.Ws = [self.rng.normal(size=(sizes[i + 1], sizes[i]))
                   * (self.init_scale or np.sqrt(2 / sizes[i]))
                   for i in range(len(sizes) - 1)]
        self.bs = [np.zeros(s) for s in sizes[1:]]
        L = len(self.Ws)

        prev = np.inf
        for epoch in range(max_epochs):
            zs, hs, P = self.forward(X)
            loss = (-np.mean(np.log(P[np.arange(n), y] + 1e-12))
                    + reg_strength / 2 * sum(np.sum(W ** 2) for W in self.Ws))
            if prev - loss < tol:
                break
            prev = loss

            D = P - Y                          # row j = delta of sample j
            gWs, gbs = [None] * L, [None] * L
            for i in reversed(range(L)):       # all grads use the old weights, then update
                gWs[i] = (hs[i].T @ D).T
                gbs[i] = D.sum(axis=0)
                if i > 0:
                    D = (D @ self.Ws[i]) * (zs[i - 1] > 0)
            for i in range(L):
                self.Ws[i] -= lr * (gWs[i] / n + reg_strength * self.Ws[i])
                self.bs[i] -= lr * gbs[i] / n
        self.epochs_run = epoch + 1

    def predict_ids(self, X):
        return np.argmax(self.forward(X)[2], axis=1)

    def predict(self, text):
        return LABELS[int(self.predict_ids(sp.csr_matrix(self.encoder(text)))[0])]

## 6. Experiments (together)


In [34]:
import time

def macro_f1(y_true, y_pred, K=7):
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    f1s = []
    for k in range(K):
        tp = np.sum((y_pred == k) & (y_true == k))
        fp = np.sum((y_pred == k) & (y_true != k))
        fn = np.sum((y_pred != k) & (y_true == k))
        f1s.append(2 * tp / (2 * tp + fp + fn) if tp + fp + fn else 0.0)
    return float(np.mean(f1s))

encoders = {
    "indicator": TokenIndicator(tokenizer),
    "bow": BagOfWords(tokenizer),
    "bow+bigrams": BagOfUnigramsAndBigrams(tokenizer, texts),
}
architectures = {
    "loglinear": lambda enc: LogLinear(enc),
    "mlp": lambda enc: MLP(enc, hidden=(100,)),
    "deep_mlp": lambda enc: MLP(enc, hidden=(128, 64)),
}

# encode every text once per representation; folds then just select rows
row_of = {t: i for i, t in enumerate(texts)}
assert len(row_of) == len(texts), "duplicate texts would break the row lookup"
y_all = np.array([y for _, y in pairs])
X_all = {rep: encode_all(enc, texts) for rep, enc in encoders.items()}
print({rep: X.shape for rep, X in X_all.items()})

results_path = results_folder / "cv_results.csv"
preds_path = results_folder / "predictions.csv"
rows = pd.read_csv(results_path).to_dict("records") if results_path.exists() else []
done = {(r["representation"], r["architecture"], r["seed"], r["fold"]) for r in rows}

for rep, enc in encoders.items():
    X = X_all[rep]
    for arch, make_model in architectures.items():
        for seed, cv in zip(seeds, cv_sets):
            for fold, split in cv.items():
                if (rep, arch, seed, fold) in done:
                    continue
                t0 = time.time()
                tr = [row_of[t] for t, _ in split["training"]]
                va = [row_of[t] for t, _ in split["validation"]]
                model = make_model(enc)
                model.train(X[tr], y_all[tr])
                y_true, y_pred = y_all[va], model.predict_ids(X[va])

                pd.DataFrame(dict(representation=rep, architecture=arch, seed=seed, fold=fold,
                                  row=va, y_true=y_true, y_pred=y_pred)
                             ).to_csv(preds_path, mode="a", header=not preds_path.exists(), index=False)
                rows.append(dict(representation=rep, architecture=arch, seed=seed, fold=fold,
                                 macro_f1=macro_f1(y_true, y_pred),
                                 accuracy=float(np.mean(y_true == y_pred)),
                                 epochs=model.epochs_run))
                pd.DataFrame(rows).to_csv(results_path, index=False)
                print(f"{rep:>12} | {arch:>9} | seed {seed} fold {fold}: "
                      f"F1 {rows[-1]['macro_f1']:.3f} ({time.time() - t0:.1f}s)")

results = pd.DataFrame(rows)
results.groupby(["representation", "architecture"])[["macro_f1", "accuracy"]].agg(["mean", "std"])

{'indicator': (7432, 1000), 'bow': (7432, 1000), 'bow+bigrams': (7432, 26015)}
   indicator | loglinear | seed 0 fold 0: F1 0.545 (8.0s)
   indicator | loglinear | seed 0 fold 1: F1 0.536 (10.6s)
   indicator | loglinear | seed 0 fold 2: F1 0.520 (10.2s)
   indicator | loglinear | seed 0 fold 3: F1 0.520 (9.0s)
   indicator | loglinear | seed 0 fold 4: F1 0.537 (9.1s)
   indicator | loglinear | seed 3 fold 0: F1 0.529 (8.7s)
   indicator | loglinear | seed 3 fold 1: F1 0.547 (8.5s)
   indicator | loglinear | seed 3 fold 2: F1 0.523 (8.9s)
   indicator | loglinear | seed 3 fold 3: F1 0.536 (8.5s)
   indicator | loglinear | seed 3 fold 4: F1 0.545 (8.4s)
   indicator | loglinear | seed 5 fold 0: F1 0.518 (8.5s)
   indicator | loglinear | seed 5 fold 1: F1 0.545 (8.6s)
   indicator | loglinear | seed 5 fold 2: F1 0.548 (8.5s)
   indicator | loglinear | seed 5 fold 3: F1 0.530 (11.1s)
   indicator | loglinear | seed 5 fold 4: F1 0.539 (12.1s)
   indicator |       mlp | seed 0 fold 0: F1 0.

macro_f1            accuracy          
                                 mean       std      mean       std
representation architecture                                        
bow            deep_mlp      0.419406  0.011835  0.427654  0.009882
               loglinear     0.534943  0.012292  0.536643  0.013324
               mlp           0.518328  0.010336  0.519510  0.011233
bow+bigrams    deep_mlp      0.448307  0.021197  0.452008  0.018863
               loglinear     0.561592  0.010160  0.562611  0.011751
               mlp           0.549319  0.011492  0.550323  0.012866
indicator      deep_mlp      0.515748  0.010257  0.522919  0.010869
               loglinear     0.534651  0.010653  0.536239  0.011659
               mlp           0.514300  0.009795  0.515788  0.010414

## 7. Bayesian comparison with a ROPE (together)
Use tutorial 

## 8. Error analysis (together)
per-class F1 and a confusion matrix